In [0]:
%run /Configurations/Init_Scripts

In [0]:
dbutils.widgets.text("job_id","-1")
JobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("run_id","-1")
PipelineRunId = dbutils.widgets.get("run_id")

ConfigID=dbutils.widgets.text("ConfigID","44")
ConfigID=dbutils.widgets.get("ConfigID")

dbutils.widgets.text('sourceTypeId','2')
sourceTypeId = dbutils.widgets.get('sourceTypeId')

dbutils.widgets.text('CreatedBy','ADB_Ingestion_SoldTo')  
CreatedBy = dbutils.widgets.get('CreatedBy')

UpdatedBy = 'ADB_Ingestion_SoldTo'  

In [0]:
database_host_url = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakeURL")
username = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakeUserName")
password = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="SnowflakePassword")
database_name = "AES_DW"
schema_name = "dw"
warehouse_name = "AES_ELT_WH_DEV"
table_name = "DIM_CUSTOMER_HIERARCHY"

In [0]:
target_table_path = '/mnt/silver/dimcustomerhierarchy'
target_table = 'silverzone.dimcustomerhierarchy'
checkpoint_column_from_target = 'SrcLastUpdatedDate'
default_date = '1900-01-01 00:00:00'

In [0]:
from pyspark.sql.utils import AnalysisException

try: 
    last_checkpoint = spark.sql(f"""
                              SELECT max({checkpoint_column_from_target}) as last_updated FROM {target_table} 
                              """).first()['last_updated']
except AnalysisException:
    last_checkpoint = default_date

In [0]:
# Read Data from the source
df_source = (spark.read
  .format("snowflake")
  # .option("dbTable", table_name)
  .option("sfUrl", database_host_url)
  .option("sfUser", username)
  .option("sfPassword", password)
  .option("sfDatabase", database_name)
  .option("sfSchema", schema_name)
  .option("sfWarehouse", warehouse_name)
  .option("query", f"""SELECT * FROM {table_name} WHERE LAST_UPDATED_DATE > TO_TIMESTAMP('{last_checkpoint}')""")
  .load()
)

In [0]:
df_source.createOrReplaceTempView("vw_source")

In [0]:

query = f'''
select DISTINCT CUSTOMER_SHIPTO_ID as ShipToAccountID, 
        CUSTOMER_SOLDTO_ID as SoldToAccountID, 
        CUSTOMER_PAYER_ID as PayerAccountID, 
        EFFECTIVE_BEGIN_DATE as EffectiveDate,
        EFFECTIVE_END_DATE as TerminationDate,
        CURRENT_FLAG as CurrentFlag,
        LAST_UPDATED_DATE as SrcLastUpdatedDate
from vw_source
where CUSTOMER_SOLDTO_ID is not null 
and CUSTOMER_PAYER_ID is not null
and SOURCE_SYSTEM_NAME = 'SAP'
'''

incremental_df = spark.sql(query)

incremental_df = incremental_df.withColumn('CreatedBy', lit(CreatedBy))\
                                .withColumn('CreatedDate', current_timestamp())\
                                .withColumn('UpdatedBy', lit(UpdatedBy))\
                                .withColumn('UpdatedDate', current_timestamp())

incremental_cols = incremental_df.columns

In [0]:
#Creating Log Entry
log_dict = {
"ConfigID" : ConfigID,
"SourceTypeID" : sourceTypeId,
"Source" : "AES_DW.DW.DIM_CUSTOMER_HIERARCHY",
"SourceFileName" : "",
"Destination" : "silverzone.dimcustomerhierarchy",
"DestinationFileName" : "",
"Run_ID": str(PipelineRunId),
"Job_ID": str(JobId)
}
audit_df = spark.createDataFrame([log_dict])

In [0]:
from delta.tables import DeltaTable

try:
    if not spark.catalog.tableExists(target_table):
        incremental_df.write.option("path", target_table_path).format("delta").saveAsTable(target_table)
    else:
        delta_target = DeltaTable.forName(spark, target_table)
        delta_target.alias("tgt").merge(
            incremental_df.alias("src"),
            """
            tgt.ShipToAccountID = src.ShipToAccountID
            AND tgt.SoldToAccountID = src.SoldToAccountID
            AND tgt.PayerAccountID = src.PayerAccountID
            AND tgt.EffectiveDate = src.EffectiveDate
            """
        ).whenMatchedUpdate(
            condition="""
            tgt.TerminationDate <> src.TerminationDate
            OR tgt.CurrentFlag <> src.CurrentFlag
            """,
            set={
                "TerminationDate": "COALESCE(src.TerminationDate, DATE_SUB(CURRENT_DATE(), 1))",
                "CurrentFlag": "src.CurrentFlag",
                "SrcLastUpdatedDate": "src.SrcLastUpdatedDate",
                "UpdatedBy": "src.UpdatedBy",
                "UpdatedDate": "CURRENT_TIMESTAMP()"
            }
        ).whenNotMatchedInsert(
            values={
                "ShipToAccountID": "src.ShipToAccountID",
                "SoldToAccountID": "src.SoldToAccountID",
                "PayerAccountID": "src.PayerAccountID",
                "EffectiveDate": "COALESCE(src.EffectiveDate, CURRENT_DATE())",
                "TerminationDate": "COALESCE(src.TerminationDate, CAST('9999-12-31' AS DATE))",
                "CurrentFlag": "src.CurrentFlag",
                "SrcLastUpdatedDate": "src.SrcLastUpdatedDate",
                "CreatedBy": "src.CreatedBy",
                "CreatedDate": "COALESCE(src.CreatedDate, CURRENT_TIMESTAMP())",
                "UpdatedBy": "src.UpdatedBy",
                "UpdatedDate": "CURRENT_TIMESTAMP()"
            }
        ).execute()
        logIntoPromotionLogTable(audit_df, CreatedBy, "Succeeded")
except Exception as e:
    logIntoPromotionLogTable(audit_df, CreatedBy, "Failed", str(e))
    raise e